In [ ]:
import pandas as pd
import requests
from io import StringIO
import openpyxl
import re

# SHFE prices

In [16]:
response = requests.get (f"https://www.shfe.com.cn/data/tradedata/future/dailydata/kx20260522.dat")

In [17]:
data = response.json()
df = pd.DataFrame(data['o_curinstrument'])

df["product_group_index"] = pd.factorize(df["PRODUCTGROUPID"])[0] # Create a new column with the group index

df = df.sort_values(["product_group_index", "DELIVERYMONTH"], ascending=[True, True], kind='stable') # Sort by group index and delivery month
df.drop(columns=["product_group_index"], inplace=True) # Remove the temporary group index column
df.reset_index(drop=True, inplace=True) # Reset the index after sorting

df.insert(0,"M",df.groupby("PRODUCTGROUPID").cumcount() + 1) # Add a new column "M" with the cumulative count of rows within each group, starting from 1
sub_totals_index = df.groupby("PRODUCTGROUPID").tail(1).index # Get the index of the last row of each group which turns to be the subtotal row
df.loc[sub_totals_index, "M"] = 0 # Set the "M" value to 0 for the subtotal rows

In [18]:
df.to_csv("shfe_prices.csv", index=False, encoding="utf-8-sig")

# SHFE stocks

In [71]:
response = requests.get(f"https://www.shfe.cn/data/tradedata/future/stockdata/weeklystock_20260522/EN/all.html")
response.content

b'<html lang="en-US">\n<head>\n    <title></title>\n    <link rel="stylesheet" type="text/css" href="./css/new.css">\n</head>\n<body>\n    <div class="comtent_table" id="stock">\n        <table class="content_intro">\n            <tr><td>Issue No.(19),2026,(Total Issues1386)\t\t\t\t  DATE:2026-05-22 </td></tr>\n            <tr><td class="expand_list_no_hidden"></td></tr>\n        </table>\n        <table class="el-table_table">\n            <thead class="is-group has-gutter">\n                <tr>\n                    <th colspan="1" rowspan="2" class="is-center is-leaf el-table__cell"><div class="cell">Region</div></th>\n                    <th colspan="1" rowspan="2" class="is-center is-leaf el-table__cell"><div class="cell">Warehouse</div></th>\n                    <th colspan="2" rowspan="1" class="is-center el-table__cell"><div class="cell">Previous Week</div></th>\n                    <th colspan="2" rowspan="1" class="is-center el-table__cell"><div class="cell">This Week</div></

In [72]:
data = pd.read_html(StringIO(response.text))

In [74]:
df = data[1] # for Copper
df.columns 

MultiIndex([(          'Region',        'Region'),
            (       'Warehouse',     'Warehouse'),
            (   'Previous Week', 'Delivery-able'),
            (   'Previous Week',    'On Warrant'),
            (       'This Week', 'Delivery-able'),
            (       'This Week',    'On Warrant'),
            (          'Change', 'Delivery-able'),
            (          'Change',    'On Warrant'),
            ('Storage Capacity',     'Last week'),
            ('Storage Capacity',     'This Week'),
            ('Storage Capacity',        'Change')],
           )

In [76]:
# Flatten the MultiIndex columns if they exist
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] if col[0] == col [1] else f"{col[0]} ({col[1]})" for col in df.columns]

In [84]:
DF = pd.DataFrame()
for df in data:
    # Flatten the MultiIndex columns if they exist and then append to the main DataFrame
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if col[0] == col [1] else f"{col[0]} ({col[1]})" for col in df.columns]
df_total = pd.concat(data, ignore_index=True)

In [86]:
with pd.ExcelWriter("shfe_stocks111.xlsx", engine="openpyxl") as writer:
    df_total.to_excel(writer, index=False, sheet_name="Stocks")

In [83]:
for df in data:
    print(df.shape, isinstance(df.columns, pd.MultiIndex))

(1, 1) False
(39, 11) False
(47, 11) False
(31, 11) False
(17, 11) False
(17, 11) False
(15, 11) False
(20, 11) False
(9, 8) False
(21, 11) False
(29, 11) False
(14, 8) False
(21, 8) False
(24, 11) False
(9, 8) False
(17, 11) False
(12, 8) False
(10, 12) False
(16, 11) False
(28, 8) False
(2, 3) False
(7, 8) False
(16, 8) False
(13, 8) False
(8, 8) False
(28, 8) False
(8, 8) False
(11, 8) False
(2, 8) False
(1, 1) False
(73, 9) False
(7, 8) False
(4, 8) False
(18, 11) False
(13, 11) False


In [3]:
tables = pd.read_html(StringIO(response.text)) # Delivers a list of dataframes, one for each table found in the HTML
tables[1]
for table in tables:
    print(table.iloc[0,0]) # Print the first cell of each table to identify which one contains the desired data

Issue No.(19),2026,(Total Issues1386) DATE:2026-05-22
COPPER
ALUMINUM
ZINC
LEAD
NICKEL
TIN
Aluminlum Oxide(warehourse)
Aluminlum Oxide(factory warehouse)
Casting Aluminium Alloy
NATURAL RUBBER
Butadiene Rubber(warehourse)
Butadiene Rubber(factory warehouse)
Pulp(warehourse)
Pulp(factory warehouse)
OFFSET PAPER(warehourse)
OFFSET PAPER(factory warehouse)
FUEL OIL
BITUMEN(warehourse)
BITUMEN(factory warehouse)
GOLD
SILVER
Rebar(warehourse)
Rebar(factory warehouse)
WIRE ROD
HOT ROLLED COILS(warehourse)
HOT ROLLED COILS(factory warehouse)
Stainless Steel(warehourse)
Stainless Steel(factory warehouse)
Note: 1. Theoretical available storage capacity is for the entire depot, but not by crude. 2. The Basra light crude oil(levelⅠ) is the Basra light that has an API greater than or equal to 28 and sulfur content lower than or equal to 3.5%. The Basra light crude oil(levelⅡ) is the Basra light that has an API greater than or equal to 28 and sulfur content between 3.5% and 4.0%(4.0% included), or 

In [4]:
def sanitize_sheet_name(name):
    name = re.sub(r'[:\\/?*\[\]]', '_', str(name))
    return name[:31]

with pd.ExcelWriter("output.xlsx", engine="openpyxl") as writer:
    for i, df in enumerate(tables, start=1):
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [" ".join(map(str, col)).strip() for col in df.columns]
        df.to_excel(writer, sheet_name=sanitize_sheet_name(f"{i}. {df.iloc[0,0]}"), index=False)

In [9]:
df.iloc[0,0]

'COPPER(BC)'

In [10]:
df = pd.DataFrame() # Create an empty dataframe to store the concatenated results
for table in tables:
    df = pd.concat([df, table], ignore_index=True) # Concatenate all tables into a single dataframe

In [12]:
table.columns

Index(['Region Region', 'Warehouse Warehouse',
       'Storage of last week Delivery-able', 'Storage of last week On Warrant',
       'Storage of this week Delivery-able', 'Storage of this week On Warrant',
       'Storage Change Delivery-able', 'Storage Change On Warrant',
       'Storage Capacity Last week', 'Storage Capacity This Week',
       'Storage Capacity Change'],
      dtype='str')

In [10]:
df["Storage of this week", "Delivery-able"]

0     COPPER(BC)
1           2362
2            272
3              0
4          11079
5           1011
6           6679
7              0
8          21403
9            100
10             0
11           100
12         21503
13    COPPER(BC)
14          2362
15           272
16             0
17         11079
18          1011
19          6679
20             0
21         21403
22           100
23             0
24           100
25         21503
Name: (Storage of this week, Delivery-able), dtype: str

In [20]:
# SHFE old version
response = requests.get(f"https://www.shfe.cn/data/tradedata/future/weeklydata/20250905weeklystock.dat")

In [21]:
data = response.json()
df = pd.DataFrame(data['o_cursor'])

In [22]:
df.columns

Index(['WHSTOCKS', 'WGHTUNIT', 'WHABBRNAME', 'SPOTWGHTS', 'ROWSTATUS',
       'PREWHSTOCKS', 'WHSTOCKCHANGE', 'REGNAME', 'PREWRTWGHTS', 'SPOTCHANGE',
       'VARNAME', 'WHROWS', 'WHTYPE', 'VARID', 'WRTWGHTS', 'WRTWEEKRPTTYPE',
       'WRTCHANGE', 'PRESPOTWGHTS'],
      dtype='str')

In [16]:
print(df["VARID"].unique())
print(df["VARNAME"].unique())

<StringArray>
['cu', 'al', 'zn', 'pb', 'ni', 'sn', 'ao', 'ru', 'br', 'sp', 'bu', 'au', 'ag',
 'rb', 'wr', 'hc', 'ss']
Length: 17, dtype: str
<StringArray>
[                                    '铜$$COPPER',
                                  '铝$$ALUMINIUM',
                                       '锌$$ZINC',
                                       '铅$$LEAD',
                                     '镍$$NICKEL',
                                        '锡$$TIN',
            '氧化铝(仓库)$$Aluminium Oxide Warehouse',
    '氧化铝(厂库)$$Aluminium Oxide Factory Warehouse',
                          '天然橡胶$$NATURAL RUBBER',
         '丁二烯橡胶(仓库)$$Butadiene Rubber warehouse',
 '丁二烯橡胶(厂库)$$Butadiene Rubber factory warehouse',
                        '纸浆(仓库)$$Pulp Warehouse',
                '纸浆(厂库)$$Pulp Factory Warehouse',
                   '石油沥青(仓库)$$BITUMEN Warehouse',
           '石油沥青(厂库)$$BITUMEN Factory Warehouse',
                                      '黄金$$GOLD',
                                    '白银$$SILV